# Netflix Stock — Story & Insights
---

## The Story

1. **The Long Climb**  (2002–2021)
2. **Year by Year**
3. **The Crash** — 2022
4. **How Risky Is Netflix?**
5. **Seasonality**

---

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Load dataset
df = pd.read_csv('datasets/netflix_stock.csv', header=[0, 1], index_col=0)
df.columns = df.columns.get_level_values(0)
df.index = pd.to_datetime(df.index)
df.index.name = 'Date'
df = df.sort_index()

# Derived columns used across all sections
df['Daily_Return']  = df['Close'].pct_change() * 100
df['MA_50']         = df['Close'].rolling(50).mean()
df['MA_200']        = df['Close'].rolling(200).mean()
df['Volatility_30'] = df['Daily_Return'].rolling(30).std()
df['Year']          = df.index.year
df['Month']         = df.index.month

print(f"Data loaded: {df.index.min().date()} to {df.index.max().date()}  |  {len(df):,} trading days")

Data loaded: 2002-05-23 to 2026-01-30  |  5,961 trading days


---
## The Long Climb

Netflix went public in May 2002. The prices here are **split-adjusted** — Netflix split its stock twice (7:1 in 2015 and 10:1 in 2022), so the numbers look smaller than what you may have seen in the news back then.

The chart shows Netflix's full price history, with major eras highlighted and the **50-day / 200-day moving averages** drawn in.

In [2]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df['Close'],
    name='Close Price',
    line=dict(color='#E50914', width=1.5),
    hovertemplate='%{x|%b %d, %Y}<br>$%{y:.2f}<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=df.index, y=df['MA_50'],
    name='50-day MA',
    line=dict(color='#F5A623', width=1.2, dash='dot'),
    hovertemplate='50-MA: $%{y:.2f}<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=df.index, y=df['MA_200'],
    name='200-day MA',
    line=dict(color='#4A90E2', width=1.2, dash='dot'),
    hovertemplate='200-MA: $%{y:.2f}<extra></extra>'
))

eras = [
    ('2002-05-23', '2010-12-31', 'rgba(100,100,100,0.08)', 'DVD Era',         0),
    ('2011-01-01', '2019-12-31', 'rgba(78,170,100,0.08)',  'Streaming Rise',   0),
    ('2020-01-01', '2021-11-17', 'rgba(255,200,0,0.10)',   'COVID',            0),
    ('2021-11-17', '2022-12-31', 'rgba(229,9,20,0.10)',    '2022 Crash',     -18),
    ('2023-01-01', '2026-12-31', 'rgba(78,170,100,0.10)',  'Recovery',       -36),
]
for x0, x1, color, label, yshift in eras:
    fig.add_vrect(x0=x0, x1=x1, fillcolor=color, line_width=0,
                  annotation_text=label, annotation_position='top left',
                  annotation=dict(font_size=11, font_color='#444', yshift=yshift))

fig.update_layout(
    title='Netflix Stock Price — Full History (2002–2026)',
    xaxis_title='Date', yaxis_title='Price (USD)',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    template='plotly_white', height=500
)
fig.show()

---
## Year by Year: Winners and Losers

This chart shows how much Netflix stock gained or lost each year.

In [3]:
yearly = df.groupby('Year')['Close'].agg(first='first', last='last')
yearly['Return_Pct'] = (yearly['last'] / yearly['first'] - 1) * 100
yearly = yearly[yearly.index >= 2003]

colors = ['#E50914' if r < 0 else '#2ECC71' for r in yearly['Return_Pct']]

fig = go.Figure(go.Bar(
    x=yearly.index.astype(str),
    y=yearly['Return_Pct'].round(1),
    marker_color=colors,
    text=yearly['Return_Pct'].round(1).astype(str) + '%',
    textposition='outside',
    hovertemplate='%{x}: %{y:.1f}%<extra></extra>'
))
fig.add_hline(y=0, line_dash='solid', line_color='black', line_width=1)

fig.update_layout(
    title='Annual Return by Year',
    xaxis_title='Year', yaxis_title='Annual Return (%)',
    template='plotly_white', height=450,
    yaxis=dict(ticksuffix='%')
)
fig.show()

**Insight:** 2013 (+297%) and 2004 (+230%) were huge years for Netflix. 2022 (−51%) was the worst year on record. Netflix lost subscribers for the first time ever. Then 2023 bounced back with +65%.

---
## The 2022 Crash Up Close

The years 2021–2023 were the wildest period in Netflix's history. The chart below zooms in on this period, with trading volume shown at the bottom.

In [4]:
focus = df['2021-01-01':'2023-12-31'].copy()

weekly = focus.resample('W').agg(
    Open=('Open', 'first'),
    High=('High', 'max'),
    Low=('Low', 'min'),
    Close=('Close', 'last'),
    Volume=('Volume', 'sum')
).dropna()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.06, row_heights=[0.7, 0.3],
    subplot_titles=['Price (Weekly Candles)', 'Volume']
)

fig.add_trace(go.Candlestick(
    x=weekly.index,
    open=weekly['Open'], high=weekly['High'],
    low=weekly['Low'],   close=weekly['Close'],
    name='NFLX',
    increasing_line_color='#2ECC71',
    decreasing_line_color='#E50914'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=weekly.index, y=weekly['Volume'] / 1e6,
    name='Volume (M)', marker_color='#4A90E2', opacity=0.6
), row=2, col=1)

peak_date   = focus['Close'].idxmax()
trough_date = focus['Close']['2022-01-01':].idxmin()

fig.add_annotation(
    x=peak_date, y=float(focus.loc[peak_date, 'High']) * 1.05,
    text=f"Peak ${focus.loc[peak_date,'Close']:.0f}",
    showarrow=True, arrowhead=2, row=1, col=1,
    font=dict(color='green', size=12)
)
fig.add_annotation(
    x=trough_date, y=float(focus.loc[trough_date, 'Low']),
    text=f"Trough ${focus.loc[trough_date,'Close']:.0f}",
    showarrow=True, arrowhead=2, ay=-40, row=1, col=1,
    font=dict(color='red', size=12)
)

fig.update_layout(
    title='The 2022 Crash — Boom, Bust, Recovery (2021–2023)',
    xaxis_rangeslider_visible=False,
    template='plotly_white', height=550, showlegend=False
)
fig.update_yaxes(title_text='Price ($)', row=1, col=1)
fig.update_yaxes(title_text='Volume (M)', row=2, col=1)
fig.show()

**Insight:** Netflix hit **$69** (split-adjusted) in November 2021, then collapsed to **$17** by May 2022 — a 77% drop in just 6 months. Trading volume jumped as people rushed to sell, then rose again during the recovery. (In pre-split prices, that was a fall from ~$700 to ~$170 — the numbers you probably saw in the news.)

---
## How Risky Is Netflix?

Two ways to look at risk: how much the stock moves day-to-day, and how that volatility has changed over time.

In [5]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Daily Return Distribution', '30-Day Rolling Volatility Over Time']
)

returns = df['Daily_Return'].dropna()

fig.add_trace(go.Histogram(
    x=returns, nbinsx=80,
    marker_color='#4A90E2', opacity=0.75, name='Daily Returns',
    hovertemplate='Return: %{x:.1f}%<br>Count: %{y}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['Volatility_30'],
    name='30-day Volatility', fill='tozeroy',
    line=dict(color='#E50914', width=1),
    fillcolor='rgba(229,9,20,0.15)',
    hovertemplate='%{x|%b %Y}: %{y:.2f}%<extra></extra>'
), row=1, col=2)

fig.update_layout(
    title='Netflix Risk Profile',
    template='plotly_white', height=420, showlegend=False
)
fig.update_xaxes(title_text='Daily Return (%)', row=1, col=1)
fig.update_yaxes(title_text='Frequency', row=1, col=1)
fig.update_xaxes(title_text='Date', row=1, col=2)
fig.update_yaxes(title_text='Std Dev (%)', row=1, col=2)
fig.show()

**Insight:** Netflix moves mostly in a small range day-to-day, but big swings happen more often than you'd expect. The worst volatility came in 2012 (earnings surprises), 2020 (COVID), and 2022 (the crash). Since 2023, things have calmed down — Netflix is behaving more like a steady, mature company now.

---
## Best and Worst Months

This heatmap shows how Netflix performed each month, year by year, since it became a streaming company.

In [6]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

pivot = df.pivot_table(values='Daily_Return', index='Year', columns='Month', aggfunc='sum')
pivot.columns = month_names
pivot = pivot[pivot.index >= 2010]

fig = px.imshow(
    pivot,
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    aspect='auto',
    title='Monthly Cumulative Return Heatmap (Streaming Era, 2010–present)',
    labels=dict(x='Month', y='Year', color='Return (%)')
)
fig.update_layout(template='plotly_white', height=500)
fig.show()

**Insight:** January and October tend to be the most volatile months because that's when Netflix reports earnings. No single month is consistently great, so trying to time the market is very hard.

---
## Summary — What We Learned

| Topic | Key Takeaway |
|---|---|
| **Long-term growth** | Over 20,000% total gain from 2002 to its peak — thanks to betting on streaming |
| **Best year** | 2013: +297%, when *House of Cards* launched and streaming really took off |
| **Worst year** | 2022: −51%, as Netflix lost subscribers for the first time ever |
| **The crash** | Stock fell 77% in just 6 months, then took about 18 months to recover |
| **Volatility** | Big swings happen regularly; things calmed down after 2023 |
| **Seasonal patterns** | January and October are the biggest months because of earnings reports |

**What's next:** The final dashboard explore price history by era, compare yearly returns, zoom into the 2022 crash, and track how volatile the stock is right now.